# 第13讲：数据合并、连接与重塑

## 课程简介

在真实的数据分析任务中，数据往往不会只存放在一张表中。  
我们经常需要把多个数据源组合起来，或者把数据从一种形状转换成另一种形状。

本讲将围绕 Pandas 中的数据合并与重塑展开，重点学习：

- 层次化索引 `MultiIndex`
- 使用 `set_index()` 和 `reset_index()` 在列与索引之间转换
- 使用 `merge()` 实现类似数据库的连接
- 使用 `join()` 基于索引合并数据
- 使用 `concat()` 沿行或列连接对象
- 使用 `combine_first()` 合并重叠数据
- 使用 `stack()`、`unstack()`、`pivot()`、`melt()` 重塑数据

掌握这些内容后，就可以处理更接近真实业务场景的多表数据。


## 1 导入库与准备示例数据

为了让本课件可以独立运行，本节会先自动创建后续示例所需的数据文件。  
示例数据会保存在当前工作目录的 `examples/` 文件夹中。


In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

np.random.seed(12345)

pd.options.display.max_rows = 20
pd.options.display.max_columns = 20
pd.options.display.max_colwidth = 80
np.set_printoptions(precision=4, suppress=True)

example_dir = Path("examples")
example_dir.mkdir(exist_ok=True)

# 创建宏观经济示例数据，供 pivot / melt 示例使用
periods = pd.period_range("1959Q1", periods=80, freq="Q")
macro = pd.DataFrame({
    "year": periods.year,
    "quarter": periods.quarter,
    "realgdp": 2500 + np.linspace(0, 4000, len(periods)) + np.random.normal(0, 80, len(periods)).cumsum(),
    "infl": np.clip(np.random.normal(3.0, 1.0, len(periods)), 0.1, None),
    "unemp": np.clip(np.random.normal(6.0, 1.2, len(periods)), 2.0, None),
    "cpi": 50 + np.linspace(0, 150, len(periods)) + np.random.normal(0, 1, len(periods)).cumsum(),
    "m1": 150 + np.linspace(0, 650, len(periods)) + np.random.normal(0, 3, len(periods)).cumsum(),
    "tbilrate": np.clip(np.random.normal(4.0, 1.0, len(periods)), 0.1, None)
})
macro.to_csv(example_dir / "macrodata.csv", index=False)

print("示例数据已准备完成：", (example_dir / "macrodata.csv").resolve())


### <font color='limegreen'><b>技巧与提示</b></font>

合并和重塑数据时，最容易出错的地方通常不是函数语法，而是**键、索引和数据形状**。  
建议在操作前先确认：

- 哪些列是连接键？
- 连接键是否唯一？
- 是按行拼接还是按列拼接？
- 结果是否应该保留所有键，还是只保留交集？
- 数据现在是长格式还是宽格式？


## 2 层次化索引

层次化索引（hierarchical indexing）也称为 `MultiIndex`，它允许一个轴上拥有多个索引级别。  
它可以帮助我们用二维表格表达更高维的数据结构。

下面先创建一个带有两层索引的 Series。


In [ ]:
data = pd.Series(np.random.uniform(size=9),
                 index=[["a", "a", "a", "b", "b", "c", "c", "d", "d"],
                        [1, 2, 3, 1, 3, 1, 2, 2, 3]])
data

结果左侧有两层索引，说明这个 Series 使用的是 `MultiIndex`。

In [ ]:
data.index

对于层次化索引对象，可以使用**部分索引**选取数据子集。

In [ ]:
data["b"]

In [ ]:
data["b":"c"]

In [ ]:
data.loc[["b", "d"]]

也可以在内层索引中进行选取。

In [ ]:
data.loc[:, 2]

层次化索引在数据重塑和分组汇总中非常常见。  
例如，可以通过 `unstack()` 将内层索引旋转为列，得到一个 DataFrame。


In [ ]:
data.unstack()

`unstack()` 的逆运算是 `stack()`。

In [ ]:
data.unstack().stack()

DataFrame 的行和列都可以使用层次化索引。

In [ ]:
frame = pd.DataFrame(np.arange(12).reshape((4, 3)),
                     index=[["a", "a", "b", "b"], [1, 2, 1, 2]],
                     columns=[["Ohio", "Ohio", "Colorado"],
                              ["Green", "Red", "Green"]])
frame

可以给索引层级设置名称。设置名称后，结果展示会更清楚。

In [ ]:
frame.index.names = ["key1", "key2"]
frame.columns.names = ["state", "color"]
frame

In [ ]:
frame.index.nlevels

有了分层列索引后，可以轻松选取某一组列。

In [ ]:
frame["Ohio"]

### 2.1 重排与分级排序

有时需要调整索引层级的顺序，或根据某个层级对数据排序。  
`swaplevel()` 可以交换两个索引级别，`sort_index()` 可以按指定级别排序。


In [ ]:
frame.swaplevel("key1", "key2")

`sort_index(level=...)` 可以根据某个索引层级排序。  
在交换层级后，通常会接着排序，让结果更易读。


In [ ]:
frame.sort_index(level=1)

In [ ]:
frame.swaplevel(0, 1).sort_index(level=0)

### 2.2 根据级别汇总统计

对于带层次化索引的 Series 或 DataFrame，可以根据某个索引层级分组汇总。  
现代 Pandas 中，推荐使用 `groupby(level=...)` 完成这类操作。


In [ ]:
frame.groupby(level="key2").sum()

In [ ]:
frame.T.groupby(level="color").sum().T


### 2.3 使用 DataFrame 的列创建索引

在实际数据中，常常需要把一列或多列转换为行索引，或者把索引重新变回普通列。  
这两个操作分别对应 `set_index()` 和 `reset_index()`。


In [ ]:
frame = pd.DataFrame({"a": range(7), "b": range(7, 0, -1),
                      "c": ["one", "one", "one", "two", "two",
                            "two", "two"],
                      "d": [0, 1, 2, 0, 1, 2, 3]})
frame

`set_index()` 可以把一个或多个列转换为行索引，并返回新的 DataFrame。

In [ ]:
frame2 = frame.set_index(["c", "d"])
frame2

默认情况下，被转换为索引的列会从 DataFrame 中移除。  
如果希望保留这些列，可以设置 `drop=False`。


In [ ]:
frame.set_index(["c", "d"], drop=False)

`reset_index()` 的功能与 `set_index()` 相反，它会把索引层级移回普通列。

In [ ]:
frame2.reset_index()

### <font color='darkorange'><b>动手练习 1</b></font>

#### 题目
请创建一个包含 `class`、`name`、`score` 三列的 DataFrame，并完成：

1. 使用 `set_index()` 将 `class` 和 `name` 设置为层次化索引
2. 按 `class` 汇总平均分
3. 使用 `reset_index()` 将索引恢复为普通列

#### 你的答案
请在下方代码单元中完成练习。


In [ ]:
# Write your code here



#### 参考答案

<details>
<summary>点击查看示例代码</summary>

```python
scores = pd.DataFrame({
    "class": ["A", "A", "B", "B"],
    "name": ["Alice", "Bob", "Cathy", "David"],
    "score": [85, 90, 78, 92]
})

scores_indexed = scores.set_index(["class", "name"])
print(scores_indexed)

print(scores_indexed.groupby(level="class").mean())

scores_indexed.reset_index()
```

</details>


## 3 合并数据集

Pandas 中常用的数据组合方法包括：

| 方法 | 主要用途 |
|---|---|
| `pd.merge()` | 根据一个或多个键连接 DataFrame，类似 SQL join |
| `DataFrame.join()` | 更方便地按索引连接 |
| `pd.concat()` | 沿某个轴堆叠或拼接多个对象 |
| `combine_first()` | 用一个对象的数据填补另一个对象中的缺失值 |

下面先学习最常见的数据库风格合并。


In [ ]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64")})
df1

In [ ]:
df2 = pd.DataFrame({"key": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})
df2

这是一个多对一合并的例子：`df1` 中有多个 `a` 和 `b`，而 `df2` 中每个键只出现一次。

In [ ]:
pd.merge(df1, df2)

如果没有显式指定连接键，`merge()` 会自动使用两个 DataFrame 中同名的列。  
不过在实际项目中，更推荐明确写出 `on=...`，避免误用不该作为键的列。


In [ ]:
pd.merge(df1, df2, on="key")

如果两个对象中的连接键列名不同，可以分别使用 `left_on` 和 `right_on` 指定。

In [ ]:
df3 = pd.DataFrame({"lkey": ["b", "b", "a", "c", "a", "a", "b"],
                    "data1": pd.Series(range(7), dtype="Int64")})
df4 = pd.DataFrame({"rkey": ["a", "b", "d"],
                    "data2": pd.Series(range(3), dtype="Int64")})

In [ ]:
df3

In [ ]:
df4

In [ ]:
pd.merge(df3, df4, left_on="lkey", right_on="rkey")

默认情况下，`merge()` 执行的是内连接（inner join），结果只保留两边共有的键。  
其他常见连接方式包括 `left`、`right` 和 `outer`。  
Left Join（左连接）：保留左侧所有行，右侧没有匹配时填充 NaN。  
Reft Join（左连接）：保留右侧所有行，左侧没有匹配时填充 NaN。

In [ ]:
pd.merge(df1, df2, how="outer")

多对多合并会产生键匹配后的笛卡尔积，因此结果行数可能明显增加。

In [ ]:
df1 = pd.DataFrame({"key": ["b", "b", "a", "c", "a", "b"],
                    "data1": pd.Series(range(6), dtype="Int64")})
df1

In [ ]:
df2 = pd.DataFrame({"key": ["a", "b", "a", "b", "d"],
                    "data2": pd.Series(range(5), dtype="Int64")})
df2

In [ ]:
pd.merge(df1, df2, on="key", how="left")

多对多连接中，相同键的行会两两组合。连接方式只决定保留哪些键。

In [ ]:
pd.merge(df1, df2, how="inner")

如果需要根据多个键合并，可以把列名列表传给 `on`、`left_on` 或 `right_on`。

In [ ]:
left = pd.DataFrame({"key1": ["foo", "foo", "bar"],
                     "key2": ["one", "two", "one"],
                     "lval": pd.Series([1, 2, 3], dtype='Int64')})
right = pd.DataFrame({"key1": ["foo", "foo", "bar", "bar"],
                      "key2": ["one", "one", "one", "two"],
                      "rval": pd.Series([4, 5, 6, 7], dtype='Int64')})
pd.merge(left, right, on=["key1", "key2"], how="outer")

多个键可以理解为共同形成一个复合连接键。  
如果合并后出现重名列，可以使用 `suffixes` 参数指定左右表列名后缀。


In [ ]:
pd.merge(left, right, on="key1")

In [ ]:
pd.merge(left, right, on="key1", suffixes=("_left", "_right"))

### <font color='darkorange'><b>动手练习 2</b></font>

#### 题目
有两个表：订单表 `orders` 和客户表 `customers`。请完成：

1. 根据 `customer_id` 合并两个表
2. 分别尝试 `inner`、`left`、`outer` 三种连接方式
3. 观察三种结果的行数和缺失值差异

#### 你的答案
请在下方代码单元中完成练习。


In [ ]:
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4],
    "customer_id": ["C1", "C2", "C2", "C4"],
    "amount": [100, 200, 150, 80]
})

customers = pd.DataFrame({
    "customer_id": ["C1", "C2", "C3"],
    "region": ["North", "South", "East"]
})

# Write your code here



#### 参考答案

<details>
<summary>点击查看示例代码</summary>

```python
orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4],
    "customer_id": ["C1", "C2", "C2", "C4"],
    "amount": [100, 200, 150, 80]
})

customers = pd.DataFrame({
    "customer_id": ["C1", "C2", "C3"],
    "region": ["North", "South", "East"]
})

print(pd.merge(orders, customers, on="customer_id", how="inner"))
print(pd.merge(orders, customers, on="customer_id", how="left"))
print(pd.merge(orders, customers, on="customer_id", how="outer"))
```

</details>


### 3.1 索引上的合并

有时连接键不是普通列，而是 DataFrame 的索引。  
这时可以使用 `left_index=True` 或 `right_index=True`。


In [ ]:
left1 = pd.DataFrame({"key": ["a", "b", "a", "a", "b", "c"],
                      "value": pd.Series(range(6), dtype="Int64")})
left1

In [ ]:
right1 = pd.DataFrame({"group_val": [3.5, 7]}, index=["a", "b"])
right1

In [ ]:
pd.merge(left1, right1, left_on="key", right_index=True)

默认连接方式仍然是内连接。  
如果希望保留两边所有键，可以设置 `how="outer"`。


In [ ]:
pd.merge(left1, right1, left_on="key", right_index=True, how="outer")

对于层次化索引，索引上的合并相当于多键合并。

In [ ]:
lefth = pd.DataFrame({"key1": ["Ohio", "Ohio", "Ohio",
                               "Nevada", "Nevada"],
                      "key2": [2000, 2001, 2002, 2001, 2002],
                      "data": pd.Series(range(5), dtype="Int64")})
righth_index = pd.MultiIndex.from_arrays(
    [
        ["Nevada", "Nevada", "Ohio", "Ohio", "Ohio", "Ohio"],
        [2001, 2000, 2000, 2000, 2001, 2002]
    ]
)
righth = pd.DataFrame({"event1": pd.Series([0, 2, 4, 6, 8, 10], dtype="Int64",
                                           index=righth_index),
                       "event2": pd.Series([1, 3, 5, 7, 9, 11], dtype="Int64",
                                           index=righth_index)})
lefth

In [ ]:
righth

当右侧对象使用层次化索引时，左侧需要提供多个列与之匹配。

In [ ]:
pd.merge(lefth, righth, left_on=["key1", "key2"], right_index=True)

In [ ]:
pd.merge(lefth, righth, left_on=["key1", "key2"],
         right_index=True, how="outer")

两个 DataFrame 也可以同时基于索引进行合并。

In [ ]:
left2 = pd.DataFrame([[1., 2.], [3., 4.], [5., 6.]],
                     index=["a", "c", "e"],
                     columns=["Ohio", "Nevada"]).astype("Int64")
left2

In [ ]:
right2 = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [13, 14]],
                      index=["b", "c", "d", "e"],
                      columns=["Missouri", "Alabama"]).astype("Int64")
right2

In [ ]:
pd.merge(left2, right2, how="outer", left_index=True, right_index=True)

`join()` 是 DataFrame 的便捷方法，常用于按索引合并。  
它的默认连接方式是左连接。


In [ ]:
left2.join(right2, how="outer")

`join()` 还可以将调用对象中的某一列与右侧 DataFrame 的索引进行匹配。

In [ ]:
left1.join(right1, on="key")

`join()` 也可以一次连接多个 DataFrame。

In [ ]:
another = pd.DataFrame([[7., 8.], [9., 10.], [11., 12.], [16., 17.]],
                       index=["a", "c", "e", "f"],
                       columns=["New York", "Oregon"])
another

In [ ]:
left2.join([right2, another])

In [ ]:
left2.join([right2, another], how="outer")

### 3.2 轴向连接

轴向连接也称为拼接或堆叠。  
对于 NumPy 数组，可以使用 `np.concatenate()`；对于 Pandas 对象，更常用的是 `pd.concat()`。


In [ ]:
arr = np.arange(12).reshape((3, 4))
arr

In [ ]:
np.concatenate([arr, arr], axis=1)

Pandas 对象带有索引和列名，因此拼接时还需要考虑：

- 另一条轴上的标签是取并集还是交集？
- 是否需要保留原对象的来源信息？
- 是否需要重新生成连续整数索引？

这些问题都可以通过 `concat()` 的参数控制。


In [ ]:
s1 = pd.Series([0, 1], index=["a", "b"], dtype="Int64")
s2 = pd.Series([2, 3, 4], index=["c", "d", "e"], dtype="Int64")
s3 = pd.Series([5, 6], index=["f", "g"], dtype="Int64")

默认情况下，`concat()` 沿 `axis=0` 连接，也就是把对象上下堆叠。

In [ ]:
pd.concat([s1, s2, s3])

如果传入 `axis="columns"`，则会沿列方向拼接，结果通常是 DataFrame。

In [ ]:
pd.concat([s1, s2, s3], axis="columns")

当另一条轴的标签不完全相同时，默认使用并集；如果设置 `join="inner"`，则只保留交集。

In [ ]:
s4 = pd.concat([s1, s3])
s4

In [ ]:
pd.concat([s1, s4], axis="columns")

In [ ]:
pd.concat([s1, s4], axis="columns", join="inner")

如果希望在连接结果中保留每个对象的来源，可以使用 `keys` 创建层次化索引。

In [ ]:
result = pd.concat([s1, s1, s3], keys=["one", "two", "three"])
result

In [ ]:
result.unstack()

沿列拼接 Series 时，`keys` 会成为结果 DataFrame 的列名。

In [ ]:
pd.concat([s1, s2, s3], axis="columns", keys=["one", "two", "three"])

同样的逻辑也适用于 DataFrame。

In [ ]:
df1 = pd.DataFrame(np.arange(6).reshape(3, 2), index=["a", "b", "c"],
                   columns=["one", "two"])
df1

In [ ]:
df2 = pd.DataFrame(5 + np.arange(4).reshape(2, 2), index=["a", "c"],
                   columns=["three", "four"])
df2

In [ ]:
pd.concat([df1, df2], axis="columns", keys=["level1", "level2"])

如果传入的是字典，字典的键会被当作 `keys`。

In [ ]:
pd.concat({"level1": df1, "level2": df2}, axis="columns")

还可以用 `names` 为新创建的层次化索引命名。

In [ ]:
pd.concat([df1, df2], axis="columns", keys=["level1", "level2"],
          names=["upper", "lower"])

有时原来的行索引没有实际含义，拼接后可以使用 `ignore_index=True` 重新生成整数索引。

In [ ]:
df1 = pd.DataFrame(np.random.standard_normal((3, 4)),
                   columns=["a", "b", "c", "d"])
df2 = pd.DataFrame(np.random.standard_normal((2, 3)),
                   columns=["b", "d", "a"])
df1

In [ ]:
df2

在这种情况下，`ignore_index=True` 会丢弃原行索引，并生成新的连续整数索引。

In [ ]:
pd.concat([df1, df2], ignore_index=True)

### `concat()` 常用参数

| 参数 | 说明 |
|---|---|
| `axis` | 指定沿行还是列拼接 |
| `join` | 指定另一条轴上的标签取并集还是交集 |
| `keys` | 为拼接来源创建层次化索引 |
| `names` | 为新索引层级命名 |
| `ignore_index` | 是否忽略原索引并生成新整数索引 |


### 3.3 合并重叠数据

有时两个对象的索引部分重叠，希望用一个对象中的值补全另一个对象中的缺失值。  
这种情况可以使用 `combine_first()`。


In [ ]:
a = pd.Series([np.nan, 2.5, 0.0, 3.5, 4.5, np.nan],
              index=["f", "e", "d", "c", "b", "a"])
b = pd.Series([0., np.nan, 2., np.nan, np.nan, 5.],
              index=["a", "b", "c", "d", "e", "f"])
a

In [ ]:
b

In [ ]:
np.where(pd.isna(a), b, a)

Series 的 `combine_first()` 会按照索引对齐数据，并用传入对象的值填补调用对象中的缺失值。

In [ ]:
a.combine_first(b)

对于 DataFrame，`combine_first()` 会同时按照行索引和列名对齐。  
可以把它理解为“用另一个表给当前表打补丁”。


In [ ]:
df1 = pd.DataFrame({"a": [1., np.nan, 5., np.nan],
                    "b": [np.nan, 2., np.nan, 6.],
                    "c": range(2, 18, 4)})
df1

In [ ]:
df2 = pd.DataFrame({"a": [5., 4., np.nan, 3., 7.],
                    "b": [np.nan, 3., 4., 6., 8.]})
df2

In [ ]:
df1.combine_first(df2)

### <font color='cornflowerblue'><b>思考题 1</b></font>

`merge()`、`join()`、`concat()` 分别适合什么场景？

#### 参考答案

<details>
<summary>点击查看解释</summary>

- `merge()`：适合根据一个或多个键列做数据库风格连接
- `join()`：适合按索引连接，写法更简洁
- `concat()`：适合沿行或列直接拼接多个对象

如果你在想“根据某个键匹配行”，通常用 `merge()`；  
如果你在想“把这些表上下或左右拼起来”，通常用 `concat()`。

</details>


## 4 数据重塑和轴向旋转

重塑（reshape）是指改变数据的排列方式，而不改变数据本身的含义。  
Pandas 中常用的重塑方法包括：

- `stack()`：将列旋转为行
- `unstack()`：将行索引层级旋转为列
- `pivot()`：将长格式数据旋转为宽格式
- `melt()`：将宽格式数据旋转为长格式

下面先从层次化索引的重塑开始。


In [ ]:
data = pd.DataFrame(np.arange(6).reshape((2, 3)),
                    index=pd.Index(["Ohio", "Colorado"], name="state"),
                    columns=pd.Index(["one", "two", "three"],
                    name="number"))
data

对该 DataFrame 使用 `stack()`，可以将列索引旋转到行索引中，得到一个 Series。

In [ ]:
result = data.stack()
result

对于层次化索引 Series，可以使用 `unstack()` 将某个索引层级旋转为列。

In [ ]:
result.unstack()

默认情况下，`unstack()` 操作最内层索引。  
也可以通过层级编号或层级名称指定要旋转的索引层级。


In [ ]:
result.unstack(level=0)

In [ ]:
result.unstack(level="state")

如果不是所有分组都有完整的索引组合，`unstack()` 可能会引入缺失值。

In [ ]:
s1 = pd.Series([0, 1, 2, 3], index=["a", "b", "c", "d"], dtype="Int64")
s2 = pd.Series([4, 5, 6], index=["c", "d", "e"], dtype="Int64")
data2 = pd.concat([s1, s2], keys=["one", "two"])
data2

In [ ]:
data2.unstack()

`stack()` 默认会过滤缺失值，因此在很多情况下 `stack()` 与 `unstack()` 可以互相还原。

In [ ]:
data2.unstack().stack()

In [ ]:
data2.unstack().stack(dropna=False)


在对 DataFrame 进行 `unstack()` 时，被旋转的索引层级会成为结果列索引中的低层级。

In [ ]:
df = pd.DataFrame({"left": result, "right": result + 5},
                  columns=pd.Index(["left", "right"], name="side"))
df

In [ ]:
df.unstack(level="state")

调用 `stack()` 时，也可以通过层级名称指定要旋转的列索引层级。

In [ ]:
df.unstack(level="state").stack(level="side")

### 4.1 将长格式旋转为宽格式

时间序列或观测数据常常以“长格式”存储：每一行是一条观测，每个变量名称存在一列中。  
这种格式适合数据库存储，但有时不方便直接分析或绘图。

下面使用宏观经济示例数据构造长格式数据。


In [ ]:
data = pd.read_csv("examples/macrodata.csv")
data = data.loc[:, ["year", "quarter", "realgdp", "infl", "unemp"]]
data.head()

In [ ]:
periods = pd.PeriodIndex(year=data.pop("year"),
                         quarter=data.pop("quarter"),
                         name="date")
data.index = periods.to_timestamp("D")

data = data.reindex(columns=["realgdp", "infl", "unemp"])
data.columns.name = "item"

long_data = (data.stack()
             .reset_index()
             .rename(columns={0: "value"}))
long_data[:10]

长格式数据中，每一行代表一个时间点与一个变量的一次观测。  
如果希望不同变量分别成为列，可以使用 `pivot()` 转换为宽格式。


In [ ]:
pivoted = long_data.pivot(index="date", columns="item",
                          values="value")
pivoted.head()

`pivot()` 的三个核心参数是：

- `index`：结果的行索引
- `columns`：结果的列标签
- `values`：填充到表格中的数值列


In [ ]:
long_data["value2"] = np.random.standard_normal(len(long_data))
long_data[:10]


如果忽略 `values` 参数，结果会使用剩余所有数值列，并生成层次化列索引。

In [ ]:
pivoted = long_data.pivot(index="date", columns="item")
pivoted.head()

In [ ]:
pivoted["value"].head()

`pivot()` 本质上可以看作 `set_index()` 加 `unstack()` 的组合。

In [ ]:
unstacked = long_data.set_index(["date", "item"]).unstack(level="item")
unstacked.head()

### 4.2 将宽格式旋转为长格式

`pd.melt()` 是 `pivot()` 的逆向操作之一。  
它会把多个列合并为一列，使数据从宽格式变成长格式。


In [ ]:
df = pd.DataFrame({"key": ["foo", "bar", "baz"],
                   "A": [1, 2, 3],
                   "B": [4, 5, 6],
                   "C": [7, 8, 9]})
df

在这个例子中，`key` 是分组标识列，`A`、`B`、`C` 是需要被合并的值列。

In [ ]:
melted = pd.melt(df, id_vars="key")
melted

使用 `pivot()` 可以把 `melt()` 后的数据重塑回宽格式。

In [ ]:
reshaped = melted.pivot(index="key", columns="variable",
                        values="value")
reshaped

由于 `pivot()` 的结果把 `key` 放到了索引中，可以使用 `reset_index()` 将其重新变成普通列。

In [ ]:
reshaped.reset_index()

可以通过 `value_vars` 指定只转换部分值列。

In [ ]:
pd.melt(df, id_vars="key", value_vars=["A", "B"])

如果没有指定 `id_vars`，`melt()` 会把传入的所有列都转换为变量和值。

In [ ]:
pd.melt(df, value_vars=["A", "B", "C"])

In [ ]:
pd.melt(df, value_vars=["key", "A", "B"])

### <font color='darkorange'><b>动手练习 3</b></font>

#### 题目
请创建一个宽格式 DataFrame，包含 `student`、`math`、`english` 三列，然后完成：

1. 使用 `melt()` 转换成长格式
2. 使用 `pivot()` 转换回宽格式
3. 观察长格式与宽格式分别适合什么分析场景

#### 你的答案
请在下方代码单元中完成练习。


In [ ]:
wide = pd.DataFrame({
    "student": ["Alice", "Bob"],
    "math": [90, 85],
    "english": [88, 92]
})

# Write your code here



#### 参考答案

<details>
<summary>点击查看示例代码</summary>

```python
wide = pd.DataFrame({
    "student": ["Alice", "Bob"],
    "math": [90, 85],
    "english": [88, 92]
})

long = pd.melt(wide, id_vars="student", var_name="subject", value_name="score")
print(long)

long.pivot(index="student", columns="subject", values="score").reset_index()
```

</details>


## 课堂小结

在本讲中，我们学习了 Pandas 中的数据合并、连接与重塑。

| 知识点 | 主要内容 |
|---|---|
| **层次化索引** | `MultiIndex`、部分索引 |
| **索引层级调整** | `swaplevel()`、`sort_index()` |
| **按层级汇总** | `groupby(level=...)` |
| **列与索引转换** | `set_index()`、`reset_index()` |
| **数据库风格合并** | `pd.merge()` |
| **索引连接** | `join()`、`left_index`、`right_index` |
| **轴向拼接** | `pd.concat()` |
| **合并重叠数据** | `combine_first()` |
| **层次索引重塑** | `stack()`、`unstack()` |
| **长宽格式转换** | `pivot()`、`melt()` |


### <font color='cornflowerblue'><b>思考题 2</b></font>

为什么在合并数据之前，应该先检查连接键是否唯一？

#### 参考答案

<details>
<summary>点击查看解释</summary>

如果连接键不唯一，合并结果可能会出现多对多匹配，导致行数成倍增加。  
这不一定是错误，但如果没有意识到，就可能造成重复统计、金额放大、样本数量异常等问题。

因此，在合并前可以检查：

```python
df["key"].is_unique
df["key"].value_counts()
```

</details>


<div class="alert alert-success">

**进一步学习资源**

- Pandas 官方文档：https://pandas.pydata.org/docs/
- Pandas 官方教程：https://pandas.pydata.org/docs/getting_started/index.html
- Pandas 速查表（Cheat Sheet）：https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf

</div>

---

*本节课到此结束，感谢大家的学习！如有疑问，请随时提问。*